In [0]:
# comment for re-commit

# F&B item recommender
Next-item ("add to cart") recommender trained on `jr_febar_catalog.sales.fnb_transactions` + `fnb_transaction_items` + `fnb_menu_items`.

* **Training task:** for every basket, hold out one item and predict it from the *rest of the cart* plus context (outlet, hour, day of week). This mirrors the app use case: given what is in the cart now, what should we suggest next?
* **Model:** sklearn pipeline (multi-hot cart encoding + one-hot outlet) -> HistGradientBoostingClassifier over the 21 menu items.
* **Serving wrapper:** MLflow pyfunc that takes `cart_items` (comma-separated item_ids), `outlet`, `hour`, `day_of_week` and returns the top-k items not already in the cart (param `top_k`, default 3).
* **Registered as:** `jr_febar_catalog.sales.fnb_item_recommender` in Unity Catalog (version 1 = first run of this notebook).

## Data sources
| Table | Grain | Used for |
| --- | --- | --- |
| `jr_febar_catalog.sales.fnb_transactions` | 1 row per POS transaction | Context: `outlet`, `transaction_ts` (hour, day of week), temporal split |
| `jr_febar_catalog.sales.fnb_transaction_items` | 1 row per line item | Basket contents (distinct `item_id` per transaction) |
| `jr_febar_catalog.sales.fnb_menu_items` | 1 row per menu item (21 items) | Item names/categories returned by the model |

## Notebook flow
1. **Setup** - install `uv` (isolated env check) and a recent MLflow.
2. **Build training samples** - leave-one-out samples per basket, temporal train/test split.
3. **EDA** - target frequency, category mix by outlet, cart sizes.
4. **Train and evaluate** - HGB classifier vs. popularity and co-occurrence baselines.
5. **Package** - refit on the full window and log an MLflow pyfunc with signature + input example.
6. **Validate** - run the logged model in a fresh `uv` environment (what a serving container does).
7. **Register** - create a Unity Catalog model version.
8. **Smoke test** - load the registered version and score sample carts; app payload example.

## Results (holdout: 2026-09-12 to 2026-11-18)
| Approach | HitRate@1 | HitRate@3 | HitRate@5 | MRR |
| --- | --- | --- | --- | --- |
| Model (HGB) | 0.118 | 0.286 | 0.423 | 0.277 |
| Co-occurrence baseline | 0.114 | 0.279 | 0.409 | 0.271 |
| Popularity baseline | 0.114 | 0.271 | 0.408 | 0.269 |

The model wins on every metric but only by about 1-1.5 points: the synthetic baskets carry little cross-item signal, so recommendations lean toward popular beers and shareables. See **Limitations and next steps** at the end.

## 1. Setup
* `uv` is needed by `mlflow.models.predict(..., env_manager="uv")` in step 6, which rebuilds the model's Python environment from scratch.
* A recent MLflow (>= 2.20) gives the `name=` argument for `log_model` and array-typed signatures.
* The Python process restarts after install, so run every cell below in order after this one.

In [0]:
%pip install -q uv "mlflow>=2.20"
dbutils.library.restartPython()

## 2. Build training samples
**Basket:** the distinct `item_id`s in a transaction (quantities are ignored, since a repeat of the same item is not a recommendation).

**Leave-one-out samples:** a basket of *n* items gives *n* rows. Each row hides one item as the `target` and keeps the other *n-1* items as `cart_items`. A 1-item basket gives an empty cart, which teaches the model what to suggest before anything has been ordered.

**Context features** come from the transaction header:
* `outlet` - Concourse Kitchen, Lounge, Dome Bar, In-Seat Service
* `hour` - 0-23 from `transaction_ts`
* `day_of_week` - Spark `dayofweek` convention: **1 = Sunday ... 7 = Saturday**. The app must send the same convention.

**Temporal split:** the latest ~10% of transaction time is held out as the test set (cutoff from `approxQuantile` on the epoch timestamp). That mirrors scoring future carts. Every row from one transaction stays on the same side of the cutoff.

**Sampling:** 30% of transactions are kept with a deterministic hash on `transaction_id`, so reruns pick the same transactions and baskets are never split. That gives about 610k rows, enough to train in memory in under 2 minutes.

In [0]:
from pyspark.sql import functions as F

CAT = "jr_febar_catalog.sales"
SEED = 42

baskets = (
    spark.table(f"{CAT}.fnb_transaction_items")
    .groupBy("transaction_id").agg(F.array_sort(F.collect_set("item_id")).alias("basket"))
    .join(spark.table(f"{CAT}.fnb_transactions").select("transaction_id", "customer_id", "outlet", "transaction_ts"), "transaction_id")
)

# Temporal cutoff: last ~10% of transaction time is the untouched test period
cutoff_epoch = baskets.select(F.col("transaction_ts").cast("long").alias("e")).approxQuantile("e", [0.9], 0.001)[0]
print("Test period starts at:", spark.sql(f"select timestamp_seconds({int(cutoff_epoch)})").first()[0])

# One sample per (basket, held-out item): target = held-out item, cart = remaining items
samples = (
    baskets.select("*", F.explode("basket").alias("target"))
    .withColumn("cart_items", F.array_join(F.array_remove("basket", F.col("target")), ","))
    .withColumn("hour", F.hour("transaction_ts"))
    .withColumn("day_of_week", F.dayofweek("transaction_ts"))  # 1=Sun..7=Sat
    .withColumn("is_test", F.col("transaction_ts").cast("long") >= cutoff_epoch)
    .select("transaction_id", "transaction_ts", "cart_items", "outlet", "hour", "day_of_week", "target", "is_test")
)

# Sample whole transactions (keeps baskets together) to a size that trains quickly in-memory
sampled = samples.where(F.abs(F.xxhash64("transaction_id")) % 100 < 30)
pdf = sampled.toPandas()
train_pdf = pdf[~pdf.is_test].reset_index(drop=True)
test_pdf = pdf[pdf.is_test].reset_index(drop=True)
print(f"samples: {len(pdf):,}  train: {len(train_pdf):,}  test: {len(test_pdf):,}")
print(f"train window: {train_pdf.transaction_ts.min()} -> {train_pdf.transaction_ts.max()}")
print(f"test window:  {test_pdf.transaction_ts.min()} -> {test_pdf.transaction_ts.max()}")
display(train_pdf.head(5))

## 3. Exploratory data analysis
* **Held-out item frequency** - how imbalanced the 21 classes are. Popular items (Dome Lager, Local IPA, Nachos) dominate, which is why popularity is a strong baseline.
* **Category mix by outlet** - checks whether outlet carries signal, for example more cocktails in the Lounge.
* **Remaining cart size** - how often the model sees empty, single-item, or larger carts.

Leakage check: every feature (other cart items, outlet, time) is known when the recommendation is made, and the target item is always removed from `cart_items`.

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

menu = spark.table(f"{CAT}.fnb_menu_items").toPandas().set_index("item_id")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
train_pdf.target.map(menu.item_name).value_counts().plot.bar(ax=axes[0], title="Held-out item frequency (train)")
pd.crosstab(train_pdf.outlet, train_pdf.target.map(menu.item_category), normalize="index").plot.bar(stacked=True, ax=axes[1], title="Category mix by outlet", legend=False)
train_pdf.cart_items.str.count(",").add((train_pdf.cart_items != "").astype(int)).value_counts().sort_index().plot.bar(ax=axes[2], title="Remaining cart size")
plt.tight_layout(); plt.show()

display(pd.DataFrame({
    "feature": ["cart_items", "outlet", "hour", "day_of_week"],
    "type": ["list of item_ids (multi-hot, 21 dims)", "categorical/nominal", "numeric integer (0-23)", "numeric integer / ordinal (1-7)"],
}))

## 4. Train and evaluate
**Pipeline** (all preprocessing lives inside the model, so callers send raw values):
* `cart_items` -> `CountVectorizer(binary=True)` with comma tokenization -> 21-dim multi-hot vector
* `outlet` -> one-hot (unknown outlets are ignored, not an error)
* `hour`, `day_of_week` -> passed through as numbers (trees handle non-linear effects)
* Classifier: `HistGradientBoostingClassifier` (multiclass over item_ids), early stopping on an internal validation split, fixed hyperparameters, `random_state=42`

**Ranking:** items already in the cart get a score of -1 before ranking, so they are never recommended.

**Metrics** (on the temporal holdout):
* `HitRate@k` - share of samples where the hidden item is in the top k
* `MRR` - mean of 1 / rank of the hidden item

**Baselines:**
* *Popularity* - global item frequency in train
* *Co-occurrence* - for each cart item, P(target | cart item), summed over the cart (falls back to popularity for an empty cart)

Parameters, model metrics, and both baselines are logged to the notebook's MLflow experiment under the run `fnb_item_recommender_hgb`.

In [0]:
import numpy as np, time
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
import mlflow

FEATURES = ["cart_items", "outlet", "hour", "day_of_week"]
PARAMS = dict(max_iter=200, learning_rate=0.1, max_leaf_nodes=31, early_stopping=True, random_state=SEED)

def make_pipeline():
    pre = ColumnTransformer([
        ("cart", CountVectorizer(token_pattern=r"[^,]+", lowercase=False, binary=True), "cart_items"),
        ("outlet", OneHotEncoder(handle_unknown="ignore"), ["outlet"]),
        ("time", "passthrough", ["hour", "day_of_week"]),
    ], sparse_threshold=0)
    return Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(**PARAMS))])

def ranked_scores(proba, classes, carts):
    """Mask items already in the cart so they can never be recommended."""
    proba = proba.copy()
    idx = {c: i for i, c in enumerate(classes)}
    for r, cart in enumerate(carts):
        for it in filter(None, cart.split(",")):
            if it in idx: proba[r, idx[it]] = -1
    return proba

def rank_metrics(scores, classes, y):
    order = np.argsort(-scores, axis=1)
    ranks = np.array([np.where(classes[order[i]] == y[i])[0][0] + 1 for i in range(len(y))])
    return {"hit_rate_at_1": np.mean(ranks <= 1), "hit_rate_at_3": np.mean(ranks <= 3),
            "hit_rate_at_5": np.mean(ranks <= 5), "mrr": np.mean(1.0 / ranks)}

X_train, y_train = train_pdf[FEATURES], train_pdf.target.values
X_test, y_test = test_pdf[FEATURES], test_pdf.target.values

mlflow.set_registry_uri("databricks-uc")
with mlflow.start_run(run_name="fnb_item_recommender_hgb") as run:
    t0 = time.time()
    pipe = make_pipeline().fit(X_train, y_train)
    train_secs = time.time() - t0
    classes = pipe.classes_

    # Model
    m_model = rank_metrics(ranked_scores(pipe.predict_proba(X_test), classes, X_test.cart_items.values), classes, y_test)

    # Baseline 1: global popularity (cart items excluded)
    pop = pd.Series(y_train).value_counts(normalize=True).reindex(classes).fillna(0).values
    m_pop = rank_metrics(ranked_scores(np.tile(pop, (len(y_test), 1)), classes, X_test.cart_items.values), classes, y_test)

    # Baseline 2: item-to-item co-occurrence, P(target | cart item) summed over cart, popularity fallback
    cidx = {c: i for i, c in enumerate(classes)}
    co = np.zeros((len(classes), len(classes)))
    for cart, t in zip(X_train.cart_items.values, y_train):
        for it in filter(None, cart.split(",")): co[cidx[it], cidx[t]] += 1
    co = co / np.maximum(co.sum(1, keepdims=True), 1)
    co_scores = np.array([sum((co[cidx[i]] for i in filter(None, c.split(","))), np.zeros(len(classes))) + 1e-6 * pop
                          for c in X_test.cart_items.values])
    m_co = rank_metrics(ranked_scores(co_scores, classes, X_test.cart_items.values), classes, y_test)

    results = pd.DataFrame({"model (HGB)": m_model, "co-occurrence baseline": m_co, "popularity baseline": m_pop}).T.round(4)
    display(results.reset_index().rename(columns={"index": "approach"}))

    mlflow.log_params({**PARAMS, "features": ",".join(FEATURES), "test_cutoff_epoch": int(cutoff_epoch), "txn_sample_pct": 30})
    mlflow.log_metrics({f"test_{k}": v for k, v in m_model.items()})
    mlflow.log_metrics({f"baseline_pop_{k}": v for k, v in m_pop.items()})
    mlflow.log_metrics({f"baseline_cooc_{k}": v for k, v in m_co.items()})
    mlflow.log_metrics({"train_seconds": train_secs, "n_train": len(X_train), "n_test": len(X_test)})
    print(f"trained in {train_secs:.0f}s, n_iter={pipe.named_steps['clf'].n_iter_}")

## 5. Refit and package for serving
**Refit:** the evaluated pipeline is refit on all sampled rows (train + holdout) with the same hyperparameters, and the number of boosting iterations is fixed to what early stopping chose. Holdout metrics above come from the pre-refit model. `n_final_fit` is logged to the same run.

**`FnbRecommender` (MLflow pyfunc)** wraps the pipeline plus a menu lookup.

Input (one row per cart):

| Column | Type | Notes |
| --- | --- | --- |
| `cart_items` | string | Comma-separated item_ids, e.g. `FB01,FB12`; empty string = empty cart; spaces are stripped |
| `outlet` | string | One of the 4 outlets; unknown values are ignored |
| `hour` | long | 0-23 |
| `day_of_week` | long | 1 = Sunday ... 7 = Saturday |

Param: `top_k` (long, default 3).

Output (one row per input row): `recommended_item_ids`, `recommended_item_names`, `recommended_item_categories` (arrays of strings) and `scores` (array of doubles, model probability; higher is better).

The model is logged with an explicit signature, an input example, and pinned `scikit-learn`, `pandas`, `numpy`, and `mlflow` versions.

In [0]:
import sklearn
from mlflow.models import infer_signature

class FnbRecommender(mlflow.pyfunc.PythonModel):
    """Input rows: cart_items (comma-separated item_ids, may be empty), outlet, hour (0-23), day_of_week (1=Sun..7=Sat).
    Param top_k (default 3). Returns top-k menu items not already in the cart."""
    def __init__(self, pipeline, menu):
        self.pipeline = pipeline
        self.menu = menu  # item_id -> {item_name, item_category, price, is_alcoholic}

    def predict(self, context, model_input, params=None):
        import numpy as np, pandas as pd
        k = int((params or {}).get("top_k", 3))
        X = pd.DataFrame({
            "cart_items": model_input["cart_items"].fillna("").astype(str).str.replace(" ", ""),
            "outlet": model_input["outlet"].fillna("").astype(str),
            "hour": model_input["hour"].astype("int64"),
            "day_of_week": model_input["day_of_week"].astype("int64"),
        })
        classes = self.pipeline.classes_
        proba = self.pipeline.predict_proba(X)
        idx = {c: i for i, c in enumerate(classes)}
        out = []
        for r, cart in enumerate(X["cart_items"]):
            p = proba[r].copy()
            for it in filter(None, cart.split(",")):
                if it in idx: p[idx[it]] = -1.0
            top = np.argsort(-p)[:k]
            ids = [str(classes[i]) for i in top]
            out.append({
                "recommended_item_ids": ids,
                "recommended_item_names": [self.menu[i]["item_name"] for i in ids],
                "recommended_item_categories": [self.menu[i]["item_category"] for i in ids],
                "scores": [float(round(p[i], 5)) for i in top],
            })
        return pd.DataFrame(out)

# Refit the same pipeline/hyperparameters on the full sampled window (train + holdout)
X_all, y_all = pdf[FEATURES], pdf.target.values
final_pipe = make_pipeline().set_params(clf__early_stopping=False, clf__max_iter=int(pipe.named_steps["clf"].n_iter_)).fit(X_all, y_all)

menu_dict = menu[["item_name", "item_category", "price", "is_alcoholic"]].to_dict(orient="index")
reco_model = FnbRecommender(final_pipe, menu_dict)

input_example = pd.DataFrame({
    "cart_items": ["FB01,FB12", "", "FB04"],
    "outlet": ["Dome Bar", "Concourse Kitchen", "Lounge"],
    "hour": [19, 13, 21],
    "day_of_week": [7, 1, 6],
})
params = {"top_k": 3}
signature = infer_signature(input_example, reco_model.predict(None, input_example, params), params=params)

with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_metrics({"n_final_fit": len(X_all)})
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=reco_model,
        signature=signature,
        input_example=(input_example, params),
        pip_requirements=[f"scikit-learn=={sklearn.__version__}", f"pandas=={pd.__version__}", f"numpy=={np.__version__}", f"mlflow=={mlflow.__version__}"],
    )
print(model_info.model_uri)
print(signature)
display(reco_model.predict(None, input_example, params))

## 6. Validate in an isolated environment
A working in-notebook prediction does not prove the model will load on a serving endpoint. This cell checks that the artifact has a signature and an input example. It then uses `uv` to install the declared requirements into a fresh virtualenv and scores the input example there. Only register the model if this cell succeeds.

In [0]:
loaded = mlflow.pyfunc.load_model(model_info.model_uri)
assert loaded.metadata.signature is not None, "missing signature"
assert loaded.input_example is not None, "missing input example"

# Installs pip_requirements from scratch, like a serving container would (top_k uses its default of 3)
mlflow.models.predict(model_uri=model_info.model_uri, input_data=input_example, env_manager="uv")

## 7. Register in Unity Catalog
Each run of this cell creates a **new version** of `jr_febar_catalog.sales.fnb_item_recommender`. The exact version number is printed and kept in `model_version`. Deploy pinned versions, or use an alias such as `@champion` when you want the app to follow upgrades without code changes.

In [0]:
MODEL_NAME = "jr_febar_catalog.sales.fnb_item_recommender"
mlflow.set_registry_uri("databricks-uc")
registered_version = mlflow.register_model(model_uri=model_info.model_uri, name=MODEL_NAME, await_registration_for=300)
model_version = registered_version.version
print(f"Registered {MODEL_NAME} version {model_version} -> models:/{MODEL_NAME}/{model_version}")

## 8. Smoke test and calling from an app
The next cell loads the registered version from Unity Catalog, the same way a batch job or an app would, and scores four sample carts.

**Batch / notebook use:** `mlflow.pyfunc.load_model("models:/jr_febar_catalog.sales.fnb_item_recommender/<version>")`

**From an app (after creating a Model Serving endpoint):** POST to `/serving-endpoints/<endpoint-name>/invocations` with a body like:

```json
{
  "dataframe_records": [
    {"cart_items": "FB01,FB12", "outlet": "Dome Bar", "hour": 19, "day_of_week": 7}
  ],
  "params": {"top_k": 3}
}
```

The response has the shape `{"predictions": [{"recommended_item_ids": [...], "recommended_item_names": [...], "recommended_item_categories": [...], "scores": [...]}]}`. In a Databricks App, add the endpoint as an app resource with **Can query** permission and call it with the Databricks SDK (`w.serving_endpoints.query(...)`).

In [0]:
uc_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{model_version}")
test_carts = pd.DataFrame({
    "cart_items": ["FB01", "FB12,FB13", "", "FB04,FB06"],
    "outlet": ["Dome Bar", "In-Seat Service", "Concourse Kitchen", "Lounge"],
    "hour": [20, 19, 12, 22],
    "day_of_week": [6, 7, 1, 5],
})
res = uc_model.predict(test_carts, params={"top_k": 5})
display(pd.concat([test_carts, res], axis=1))

## Limitations and next steps
* **Weak signal:** the model beats the popularity baseline by only ~1.5 points of HitRate@3. Most of its value comes from excluding items already in the cart and from small outlet and time effects.
* **Not personalized:** `customer_id` is not used, so two guests with the same cart get the same suggestions. Adding a customer's past item affinities (computed as point-in-time features) is the most promising improvement.
* **No business rules:** the model can suggest alcohol at any outlet or hour, and it ignores price and margin. Filter or re-rank in the app, or add those rules to the wrapper, if needed.
* **Static menu:** item names and categories are frozen into the model at training time, and new menu items can't be recommended until the model is retrained.
* **Retraining:** rerun this notebook end to end (for example as a scheduled job). It logs a new MLflow run and registers a new UC version; the serving endpoint then needs to be pointed at that version or alias.